# CosmoPower-JAX emulator tutorial

This notebook demonstrates the CosmoPower-JAX emulators available in `cloelib`.

**Covered:**
- Validation of linear P(k) against CAMB
- P(k) and P_cb(k) comparison (total matter vs CDM+baryons)
- ΛCDM vs w0waCDM comparison
- Effect of neutrino mass (0, 1, 3 species)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cloelib.cosmology.camb_cosmology import (
    CAMBBackground,
    CAMBLinearPerturbations,
    CAMBNonLinearPerturbations,
)
from cloelib.cosmology.cosmopower_jax_cosmology import (
    CosmoPowerJAXLCDMPerturbations,
    CosmoPowerJAXw0waCDMPerturbations,
)

# Common settings
zs  = np.array([0.0, 0.5, 1.0, 2.0])
ks  = np.logspace(-4, 1, 300)  # Mpc^-1

# Fiducial cosmology (Planck 2018-like)
cosmo = dict(
    H0=67.0,
    Omega_b0=0.049,
    Omega_cdm0=0.270,
    Omega_k0=0.0,
    As=2.1e-9,
    ns=0.96,
    mnu=0.06,
    w0=-1.0,
    wa=0.0,
    gamma_MG=0.0,
    N_mnu=1,
)
print('Setup complete')

## 1. Validation: CosmoPowerJAX linear P(k) vs CAMB

We compare the linear matter power spectrum predicted by the CosmoPower-JAX emulator
against CAMB at several redshifts. We also show the relative difference.

In [ ]:
# CAMB
bg_camb = CAMBBackground(**cosmo)
lin_camb = CAMBLinearPerturbations(background=bg_camb, redshifts=zs)

# CosmoPowerJAX (LCDM, 1 massive neutrino)
bg_jax = CAMBBackground(**cosmo)
lin_jax = CosmoPowerJAXLCDMPerturbations.Linear(background=bg_jax, redshifts=zs)

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(zs)))

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

for i, (z, c) in enumerate(zip(zs, colors)):
    pk_camb = np.squeeze(lin_camb.matter_power_spectrum(z, ks))
    pk_jax  = np.squeeze(lin_jax.matter_power_spectrum(z, ks))
    axes[0].loglog(ks, pk_camb, color=c, lw=2,           label=f'CAMB z={z}')
    axes[0].loglog(ks, pk_jax,  color=c, lw=1.5, ls='--', label=f'CPJ  z={z}')
    axes[1].semilogx(ks, (pk_jax - pk_camb) / pk_camb * 100, color=c, label=f'z={z}')

axes[0].set_ylabel(r'$P(k)\;[\mathrm{Mpc}^3]$', fontsize=13)
axes[0].legend(fontsize=8, frameon=False, ncol=2)
axes[0].set_title('Linear P(k): CAMB (solid) vs CosmoPowerJAX (dashed)', fontsize=13)

axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].axhspan(-1, 1, alpha=0.1, color='green', label='1% band')
axes[1].set_xlabel(r'$k\;[\mathrm{Mpc}^{-1}]$', fontsize=13)
axes[1].set_ylabel(r'$(P_\mathrm{CPJ} - P_\mathrm{CAMB})\,/\,P_\mathrm{CAMB}\;[\%]$', fontsize=11)
axes[1].legend(fontsize=9, frameon=False)
axes[1].set_ylim(-5, 5)

plt.tight_layout()
plt.show()

## 2. Total matter P(k) vs CDM+baryons P_cb(k)

With massive neutrinos the total matter power spectrum P(k) is suppressed on small scales
because neutrinos free-stream and do not cluster. P_cb(k) — the CDM+baryon spectrum —
is less suppressed and is often used as input to nonlinear models (HMcode, halofit).

In [ ]:
zs_cb = np.array([0.0, 0.5, 1.0])

lin_tot = CosmoPowerJAXLCDMPerturbations.Linear(background=bg_jax,   redshifts=zs_cb)
lin_cb  = CosmoPowerJAXLCDMPerturbations.LinearCB(background=bg_jax, redshifts=zs_cb)

colors_cb = plt.cm.plasma(np.linspace(0.1, 0.85, len(zs_cb)))

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

for z, c in zip(zs_cb, colors_cb):
    pk_tot = np.squeeze(lin_tot.matter_power_spectrum(z, ks))
    pk_cb  = np.squeeze(lin_cb.matter_power_spectrum(z, ks))
    axes[0].loglog(ks, pk_tot, color=c, lw=2,           label=rf'$P_m$   z={z}')
    axes[0].loglog(ks, pk_cb,  color=c, lw=1.5, ls='--', label=rf'$P_{{cb}}$ z={z}')
    axes[1].semilogx(ks, pk_cb / pk_tot, color=c, label=f'z={z}')

axes[0].set_ylabel(r'$P(k)\;[\mathrm{Mpc}^3]$', fontsize=13)
axes[0].legend(fontsize=9, frameon=False, ncol=2)
axes[0].set_title(rf'$P_m$ (solid) vs $P_{{cb}}$ (dashed), $m_\nu={cosmo["mnu"]}$ eV', fontsize=13)

axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xlabel(r'$k\;[\mathrm{Mpc}^{-1}]$', fontsize=13)
axes[1].set_ylabel(r'$P_{cb}\,/\,P_m$', fontsize=13)
axes[1].legend(fontsize=9, frameon=False)

plt.tight_layout()
plt.show()

## 3. ΛCDM vs w0waCDM

We compare the linear matter power spectrum for a ΛCDM cosmology (w=-1, wa=0)
against two dark energy models: w0waCDM with w0=-0.9 and w0=-0.7 (both wa=0).
The ratio shows the dark energy signature in the power spectrum.

In [ ]:
z_de = np.array([0.0, 1.0, 2.0])

# ΛCDM (w0=-1, wa=0)
cosmo_lcdm = {**cosmo, 'w0': -1.0, 'wa': 0.0}
bg_lcdm    = CAMBBackground(**cosmo_lcdm)
lin_lcdm   = CosmoPowerJAXLCDMPerturbations.Linear(background=bg_lcdm, redshifts=z_de)

# w0waCDM models
de_models = [
    {'w0': -0.9, 'wa':  0.0, 'label': r'$w_0=-0.9,\,w_a=0$',   'color': 'royalblue'},
    {'w0': -0.7, 'wa':  0.0, 'label': r'$w_0=-0.7,\,w_a=0$',   'color': 'crimson'},
    {'w0': -1.0, 'wa':  0.5, 'label': r'$w_0=-1.0,\,w_a=0.5$', 'color': 'darkorange'},
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

z_ref = 0.0
pk_lcdm_ref = np.squeeze(lin_lcdm.matter_power_spectrum(z_ref, ks))
axes[0].loglog(ks, pk_lcdm_ref, 'k-', lw=2, label=r'$\Lambda$CDM z=0')

for m in de_models:
    cosmo_de = {**cosmo, 'w0': m['w0'], 'wa': m['wa']}
    bg_de    = CAMBBackground(**cosmo_de)
    lin_de   = CosmoPowerJAXw0waCDMPerturbations.Linear(background=bg_de, redshifts=z_de)
    pk_de    = np.squeeze(lin_de.matter_power_spectrum(z_ref, ks))
    axes[0].loglog(ks, pk_de, color=m['color'], lw=1.5, ls='--', label=m['label'] + ' z=0')
    axes[1].semilogx(ks, pk_de / pk_lcdm_ref, color=m['color'], lw=1.5, label=m['label'])

axes[0].set_xlabel(r'$k\;[\mathrm{Mpc}^{-1}]$', fontsize=13)
axes[0].set_ylabel(r'$P(k)\;[\mathrm{Mpc}^3]$', fontsize=13)
axes[0].set_title(r'$\Lambda$CDM vs $w_0w_a$CDM at $z=0$', fontsize=13)
axes[0].legend(fontsize=9, frameon=False)

axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xlabel(r'$k\;[\mathrm{Mpc}^{-1}]$', fontsize=13)
axes[1].set_ylabel(r'$P_{w_0w_a}\,/\,P_{\Lambda}$', fontsize=13)
axes[1].set_title('Ratio w.r.t. ΛCDM at z=0', fontsize=13)
axes[1].legend(fontsize=9, frameon=False)

plt.tight_layout()
plt.show()

## 4. Effect of neutrino mass

Massive neutrinos suppress the matter power spectrum on scales below their free-streaming
length. We compare three cases:
- **0 massive neutrinos** (massless limit)
- **1 massive neutrino** with total mass 0.06 eV
- **3 degenerate massive neutrinos** with total mass 0.3 eV

The suppression is approximately Δ P/P ≈ -8 Ω_ν / Ω_m on small scales.

In [ ]:
z_nu  = np.array([0.0, 1.0])

nu_models = [
    {'N_mnu': 0, 'mnu': 0.0,  'label': r'$N_\nu=0$, $\Sigma m_\nu=0$ eV',    'color': 'black'},
    {'N_mnu': 1, 'mnu': 0.06, 'label': r'$N_\nu=1$, $\Sigma m_\nu=0.06$ eV', 'color': 'royalblue'},
    {'N_mnu': 1, 'mnu': 0.15, 'label': r'$N_\nu=1$, $\Sigma m_\nu=0.15$ eV', 'color': 'darkorange'},
    {'N_mnu': 3, 'mnu': 0.3,  'label': r'$N_\nu=3$, $\Sigma m_\nu=0.3$ eV',  'color': 'crimson'},
]

# Reference: massless
cosmo_ref = {**cosmo, 'N_mnu': 0, 'mnu': 0.0}
bg_ref    = CAMBBackground(**cosmo_ref)
lin_ref   = CosmoPowerJAXLCDMPerturbations.Linear(background=bg_ref, redshifts=z_nu)
pk_ref    = np.squeeze(lin_ref.matter_power_spectrum(0.0, ks))

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

for m in nu_models:
    cosmo_nu = {**cosmo, 'N_mnu': m['N_mnu'], 'mnu': m['mnu']}
    bg_nu    = CAMBBackground(**cosmo_nu)
    lin_nu   = CosmoPowerJAXLCDMPerturbations.Linear(background=bg_nu, redshifts=z_nu)
    pk_nu    = np.squeeze(lin_nu.matter_power_spectrum(0.0, ks))
    axes[0].loglog(ks, pk_nu, color=m['color'], lw=1.8, label=m['label'])
    axes[1].semilogx(ks, (pk_nu - pk_ref) / pk_ref * 100, color=m['color'], lw=1.8, label=m['label'])

axes[0].set_ylabel(r'$P(k)\;[\mathrm{Mpc}^3]$', fontsize=13)
axes[0].set_title('Effect of neutrino mass on linear P(k) at z=0', fontsize=13)
axes[0].legend(fontsize=10, frameon=False)

axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel(r'$k\;[\mathrm{Mpc}^{-1}]$', fontsize=13)
axes[1].set_ylabel(r'$(P_\nu - P_0)\,/\,P_0\;[\%]$', fontsize=13)
axes[1].legend(fontsize=10, frameon=False)

plt.tight_layout()
plt.show()